# Erased or Hidden? HF Jobs launcher (runs on a free CPU Colab, no GPU needed)

This notebook does not train anything itself. It:
1. copies your finished results from Google Drive (`MyDrive/eoh_out`) to a **private** Hugging Face dataset,
2. starts the experiment as a Hugging Face Job on a rented GPU (billed per minute, stops by itself when done),
3. lets you watch the logs, and copies the results back to Drive at the end.

The Job resumes from the uploaded results: finished stages are skipped, and only A1 (new, stronger
unlearning settings), its evaluation, relearning, probes and the report are run.

**Before you start:**
- Runtime > Change runtime type > **CPU** is fine.
- Put `erased_or_hidden.py` (the latest version) in your Drive at `MyDrive/erased_or_hidden.py`.
- Token: a Hugging Face token of type **Write** (huggingface.co/settings/tokens), saved as a Colab secret named
  `HF_TOKEN` (key icon in the left bar, "Notebook access" on). Never paste it into a cell you share.
- Your Hugging Face account must have accepted the Tiny Aya licence (already done if the main notebook ran).

In [ ]:
%pip install -q -U huggingface_hub

In [ ]:
ORG = "ValueShift"                        # your Hugging Face organization
RESULTS_REPO = f"{ORG}/eoh-results"       # private dataset that holds all outputs
NAMESPACE = ORG                           # who pays for the Job: ORG, or None to bill your own account
FLAVOR = "l4x1"                           # 1x L4 (24 GB), about $0.80/hour. "t4-small" is about $0.40/hour but slower
TIMEOUT = "5h"                            # the Job is killed after this; results are pushed after every step
DRIVE_OUT = "/content/drive/MyDrive/eoh_out"
SCRIPT = "/content/drive/MyDrive/erased_or_hidden.py"

## 1. Mount Drive and log in

In [ ]:
import os
from getpass import getpass
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

try:
    from google.colab import userdata
    tok = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    tok = ""
if not tok:
    tok = getpass("Paste your Hugging Face WRITE token: ").strip()
os.environ["HF_TOKEN"] = tok  # the `hf jobs` command below reads it from here

from huggingface_hub import HfApi, whoami

me = whoami(token=tok)
orgs = [o["name"] for o in me.get("orgs", [])]
print("logged in as", me["name"], "| orgs:", orgs)
if ORG not in orgs:
    raise SystemExit(f"Your account is not a member of {ORG}. Ask the org admin to add you, or set ORG to your username.")
assert Path(SCRIPT).exists(), f"Upload erased_or_hidden.py to your Drive first (expected at {SCRIPT})"
assert Path(DRIVE_OUT, "results/layers.json").exists(), f"No finished results found in {DRIVE_OUT}"
print("found script and results:", sorted(p.name for p in Path(DRIVE_OUT).iterdir()))

## 2. Copy the finished results from Drive to the private dataset
Uploads only what changed, so it is safe to run again.

In [ ]:
api = HfApi(token=tok)
api.create_repo(RESULTS_REPO, repo_type="dataset", private=True, exist_ok=True)  # always private
info = api.repo_info(RESULTS_REPO, repo_type="dataset")
assert info.private, f"{RESULTS_REPO} is public: make it private in its settings before uploading"
api.upload_folder(repo_id=RESULTS_REPO, repo_type="dataset", folder_path=DRIVE_OUT,
                  commit_message="results from Colab (Drive)")
files = api.list_repo_files(RESULTS_REPO, repo_type="dataset")
print(len(files), "files in", RESULTS_REPO, "| e.g.", [f for f in files if f.startswith("results/")][:6])

## 3. Start the Job
Prints a Job ID. Copy it into `JOB_ID` in the next cell. You can close this tab: the Job keeps running.
If this fails with a billing or permission error, the org has no credit: set `NAMESPACE = None` in the
settings cell (bills your own account) and run this cell again.

In [ ]:
ns = f"--namespace {NAMESPACE}" if NAMESPACE else ""
!hf jobs uv run --flavor {FLAVOR} {ns} --secrets HF_TOKEN --timeout {TIMEOUT} --detach "{SCRIPT}" --stages all --results-repo {RESULTS_REPO}

## 4. Watch it
What a good start looks like: `[hub] pulled ValueShift/eoh-results`, then `[data] already done`,
`[baseline] already done`, `[layers] already done`, `[unlearn] A1: settings changed ... redoing it`,
`[unlearn] A2 already done`.
If it starts translating to Marathi or redoing A2, the upload went wrong: cancel it (last cell) and tell me.

In [ ]:
JOB_ID = ""  # paste the ID printed above
!hf jobs logs {JOB_ID} {ns} --tail 40

In [ ]:
!hf jobs inspect {JOB_ID} {ns}

## 5. When it has finished: copy the results back to Drive

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(RESULTS_REPO, repo_type="dataset", local_dir=DRIVE_OUT, token=tok)
print("results are back in", DRIVE_OUT, "- see results/summary_RE.csv and plots/")

## Stop a Job (stops the billing)

In [ ]:
!hf jobs cancel {JOB_ID} {ns}